# VLM Sign Recheck Experiment

This notebook uses the normal detector/rule pipeline first, then asks Qwen2.5-VL to read visible traffic signs and confirm whether the rule-based violation candidate is actually supported. It is meant for cases where object detection finds a sign-like object but cannot understand the sign text/symbol meaning.

In [ ]:
from pathlib import Path
from datetime import datetime
import csv
import json
import re
import sys
import traceback

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'main.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

DATASET_DIR = PROJECT_ROOT / 'dataset' / 'image'
SCENE_CONFIG = PROJECT_ROOT / 'configs' / 'scene_config.example.json'
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
EXPERIMENT_DIR = PROJECT_ROOT / 'outputs' / 'vlm_sign_recheck' / RUN_ID
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Experiment dir:', EXPERIMENT_DIR)

## Settings

Start with a small `MAX_IMAGES` because the VLM is much slower than the rule pipeline. Use a GPU runtime in Colab.

In [ ]:
MAX_IMAGES = 10
TASK_FILTER = 'parking'
THRESHOLD = 0.25
TEXT_THRESHOLD = 0.20
SAVE_ANNOTATIONS = True

QWEN_MODEL = 'Qwen/Qwen2.5-VL-3B-Instruct'
VLM_MAX_NEW_TOKENS = 512

print({
    'max_images': MAX_IMAGES,
    'task_filter': TASK_FILTER,
    'threshold': THRESHOLD,
    'text_threshold': TEXT_THRESHOLD,
    'qwen_model': QWEN_MODEL,
})

## Collect Images

In [ ]:
def infer_expected_label(image_path: Path) -> str:
    parts = set(image_path.relative_to(DATASET_DIR).parts)
    if 'not_violate' in parts:
        return 'not_violate'
    if 'violate' in parts:
        return 'violate'
    return 'unknown'

def infer_task(image_path: Path) -> str:
    rel_parts = image_path.relative_to(DATASET_DIR).parts
    return rel_parts[0] if rel_parts else 'unknown'

image_paths = sorted(
    p for p in DATASET_DIR.rglob('*')
    if p.suffix.lower() in {'.png', '.jpg', '.jpeg'}
)

records = []
for path in image_paths:
    expected_label = infer_expected_label(path)
    task = infer_task(path)
    if expected_label == 'unknown':
        continue
    if TASK_FILTER and task != TASK_FILTER:
        continue
    records.append({
        'image_path': path,
        'relative_path': str(path.relative_to(PROJECT_ROOT)),
        'task': task,
        'expected_label': expected_label,
    })

if MAX_IMAGES is not None:
    records = records[:MAX_IMAGES]

print('Images selected:', len(records))
for row in records:
    print(row['expected_label'], row['relative_path'])

## Load Pipeline And Qwen

This loads GroundingDINO through `TrafficViolationPipeline` and Qwen through `QwenVLReasoner`. The model load can take several minutes the first time.

In [ ]:
from illegal_detector.traffic_violation_pipeline import TrafficViolationPipeline
from illegal_detector.vlm_reasoner import QwenVLReasoner

try:
    import torch
    import transformers
    print('torch:', torch.__version__)
    print('transformers:', transformers.__version__)
    print('cuda available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('gpu:', torch.cuda.get_device_name(0))
        free_bytes, total_bytes = torch.cuda.mem_get_info()
        print(f'gpu memory free/total: {free_bytes / 1024**3:.2f} / {total_bytes / 1024**3:.2f} GiB')
except Exception as exc:
    print('Could not inspect torch/transformers environment:', exc)

pipeline = TrafficViolationPipeline(
    threshold=THRESHOLD,
    text_threshold=TEXT_THRESHOLD,
)

try:
    vlm_reasoner = QwenVLReasoner(
        model_id=QWEN_MODEL,
        max_new_tokens=VLM_MAX_NEW_TOKENS,
        torch_dtype='float16',
        device_map='auto',
    )
except ImportError as exc:
    raise RuntimeError(
        'VLM dependencies are missing. In Colab, run this in the first cell, restart runtime, then rerun:\n'
        '!pip install -U transformers accelerate qwen-vl-utils torch pillow tqdm pandas numpy scikit-learn matplotlib'
    ) from exc
except Exception as exc:
    message = str(exc)
    hints = [
        'VLM model failed to load/run.',
        'Use Colab Runtime > Change runtime type > GPU.',
        'Run: !pip install -U transformers accelerate qwen-vl-utils torch',
        'Restart runtime after installing packages.',
        'If this is CUDA out of memory, reduce MAX_IMAGES, restart runtime, or use a larger GPU.',
    ]
    raise RuntimeError('\n'.join(hints) + '\n\nOriginal error:\n' + message) from exc

print('Models loaded')

## VLM Prompt

The prompt forces Qwen to read sign text/symbols first, then decide whether the rule candidate should be confirmed, rejected, or left uncertain.

In [ ]:
NO_VIOLATION = 'no_violation_or_insufficient_evidence'

def rule_prediction(result: dict) -> tuple[str, str, float]:
    candidates = [v for v in result.get('violations', []) if v.get('type') != NO_VIOLATION]
    if not candidates:
        return 'not_violate', NO_VIOLATION, 0.0
    best = max(candidates, key=lambda item: item.get('confidence', 0.0))
    return 'violate', best.get('type', 'unknown'), float(best.get('confidence', 0.0))

def compact_result_for_vlm(result: dict) -> dict:
    return {
        'image_path': result.get('image_path'),
        'image_size': result.get('image_size'),
        'detections': result.get('detections', []),
        'facts': result.get('facts', {}),
        'rule_engine_candidates': result.get('violations', []),
    }

def build_sign_recheck_prompt(result: dict) -> str:
    payload = compact_result_for_vlm(result)
    return f"""
You are verifying a Vietnamese traffic violation from one image.

The object detector and rule engine may over-trigger because they can detect a sign-shaped object but cannot read the sign meaning. Your only job is to inspect visible traffic sign content: readable sign text, sign symbols, and sign restrictions.
Do not use vehicle position, lane markings, lane arrows, traffic lights, stop lines, detector boxes, or general scene layout to create a new violation decision.
Use the rule-engine output only as the candidate that may be confirmed or contradicted by readable sign content.

Decision rules:
- If a readable relevant sign clearly supports the rule candidate, keep/confirm the rule prediction.
- If a readable relevant sign clearly contradicts the rule candidate, correct the prediction.
- If the sign is unreadable, irrelevant, or absent, return keep_rule instead of guessing.
- For illegal parking, use only no-parking/no-stopping/parking-allowed sign content.
- For wrong lane, use only lane-use signs or vehicle-type restriction signs.
- Do not inspect traffic light state for this VLM step; this VLM step is sign-content-only.

Return strict JSON only, with this schema:
{{
  "visible_signs": [
    {{"description": "...", "readable_text": "...", "meaning": "...", "confidence": 0.0}}
  ],
  "sign_evidence": "supports_rule_prediction | contradicts_rule_prediction | no_readable_relevant_sign | uncertain",
  "corrected_label": "violate | not_violate | keep_rule",
  "corrected_violation_type": "illegal_parking | wrong_lane | crossing_red_light | no_violation_or_insufficient_evidence | keep_rule",
  "confidence": 0.0,
  "reason": "short explanation focused only on readable traffic sign content"
}}

Detector and rule output:
{json.dumps(payload, ensure_ascii=False, indent=2)}
""".strip()

def parse_json_from_vlm(text: str) -> dict:
    cleaned = text.strip()
    cleaned = re.sub(r'^```(?:json)?', '', cleaned).strip()
    cleaned = re.sub(r'```$', '', cleaned).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        match = re.search(r'\{.*\}', cleaned, flags=re.DOTALL)
        if match:
            return json.loads(match.group(0))
        raise

## Run Sign Recheck

In [ ]:
raw_dir = EXPERIMENT_DIR / 'pipeline_results'
vlm_dir = EXPERIMENT_DIR / 'vlm_results'
annotation_dir = EXPERIMENT_DIR / 'annotations'
raw_dir.mkdir(parents=True, exist_ok=True)
vlm_dir.mkdir(parents=True, exist_ok=True)
if SAVE_ANNOTATIONS:
    annotation_dir.mkdir(parents=True, exist_ok=True)

rows = []

for index, item in enumerate(records, start=1):
    image_path = item['image_path']
    stem = image_path.relative_to(DATASET_DIR).with_suffix('').as_posix().replace('/', '__')
    raw_json_path = raw_dir / f'{stem}.json'
    vlm_json_path = vlm_dir / f'{stem}.json'
    annotate_path = annotation_dir / f'{stem}.png' if SAVE_ANNOTATIONS else None

    row = {
        'run_id': RUN_ID,
        'image_path': item['relative_path'],
        'task': item['task'],
        'expected_label': item['expected_label'],
        'status': 'ok',
        'error': '',
        'pipeline_json': str(raw_json_path.relative_to(PROJECT_ROOT)),
        'vlm_json': str(vlm_json_path.relative_to(PROJECT_ROOT)),
    }

    try:
        result = pipeline.analyze(
            image_path=str(image_path),
            scene_config_path=str(SCENE_CONFIG),
            annotate_path=str(annotate_path) if annotate_path else None,
            segment_dir=None,
            run_vlm=False,
        )
        raw_json_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')

        rule_label, rule_type, rule_confidence = rule_prediction(result)
        prompt = build_sign_recheck_prompt(result)
        vlm_text = vlm_reasoner.reason(image_path=str(image_path), prompt=prompt)
        vlm_payload = parse_json_from_vlm(vlm_text)
        vlm_json_path.write_text(json.dumps({
            'raw_text': vlm_text,
            'parsed': vlm_payload,
        }, ensure_ascii=False, indent=2), encoding='utf-8')

        sign_evidence = str(vlm_payload.get('sign_evidence', 'uncertain')).strip().lower()
        requested_label = str(vlm_payload.get('corrected_label', 'keep_rule')).strip().lower()
        if sign_evidence == 'contradicts_rule_prediction' and requested_label in {'violate', 'not_violate'}:
            corrected_label = requested_label
        else:
            corrected_label = rule_label

        row.update({
            'rule_label': rule_label,
            'rule_type': rule_type,
            'rule_confidence': rule_confidence,
            'vlm_label': requested_label,
            'vlm_type': vlm_payload.get('corrected_violation_type', 'keep_rule'),
            'vlm_confidence': vlm_payload.get('confidence', 0.0),
            'vlm_supported_rule': sign_evidence == 'supports_rule_prediction',
            'vlm_sign_evidence': sign_evidence,
            'corrected_label': corrected_label,
            'rule_correct': rule_label == item['expected_label'],
            'corrected_correct': corrected_label == item['expected_label'],
            'vlm_reason': vlm_payload.get('reason', ''),
        })
    except Exception as exc:
        row.update({
            'rule_label': 'error',
            'rule_type': 'error',
            'rule_confidence': 0.0,
            'vlm_label': 'error',
            'vlm_type': 'error',
            'vlm_confidence': 0.0,
            'vlm_supported_rule': None,
            'vlm_sign_evidence': 'error',
            'corrected_label': 'error',
            'rule_correct': False,
            'corrected_correct': False,
            'vlm_reason': '',
            'status': 'error',
            'error': ''.join(traceback.format_exception_only(type(exc), exc)).strip(),
        })

    rows.append(row)
    print(
        f"{index:03d}/{len(records):03d} {item['relative_path']} "
        f"rule={row['rule_label']} vlm={row['vlm_label']} corrected={row['corrected_label']} status={row['status']}"
    )
    if row['status'] == 'error':
        print('  ', row['error'])

print('Done:', len(rows))

## Save Summary And Metrics

In [ ]:
summary_csv = EXPERIMENT_DIR / 'summary.csv'
fieldnames = [
    'run_id', 'image_path', 'task', 'expected_label', 'rule_label', 'rule_type',
    'rule_confidence', 'vlm_label', 'vlm_type', 'vlm_confidence',
    'vlm_supported_rule', 'vlm_sign_evidence', 'corrected_label', 'rule_correct', 'corrected_correct',
    'vlm_reason', 'status', 'error', 'pipeline_json', 'vlm_json',
]

with summary_csv.open('w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

ok_rows = [row for row in rows if row['status'] in {'ok', 'vlm_error'}]
rule_accuracy = sum(row['rule_correct'] for row in ok_rows) / len(ok_rows) if ok_rows else 0.0
corrected_accuracy = sum(row['corrected_correct'] for row in ok_rows) / len(ok_rows) if ok_rows else 0.0

print('Saved:', summary_csv)
print(f'Rule accuracy: {rule_accuracy:.3f}')
print(f'VLM-corrected accuracy: {corrected_accuracy:.3f}')

print('\nRows where VLM changed the rule label:')
for row in ok_rows:
    if row['rule_label'] != row['corrected_label']:
        print(row['image_path'], 'expected=', row['expected_label'], 'rule=', row['rule_label'], 'vlm=', row['vlm_label'])
        print(' ', row['vlm_reason'])

## Inspect VLM Decisions

In [ ]:
from IPython.display import display
from PIL import Image

for row in rows[:5]:
    print('\n', row['image_path'])
    print('expected:', row['expected_label'], 'rule:', row['rule_label'], row['rule_type'], 'vlm:', row['vlm_label'], row['vlm_type'])
    print('reason:', row['vlm_reason'])
    display(Image.open(PROJECT_ROOT / row['image_path']))
    vlm_path = PROJECT_ROOT / row['vlm_json']
    if vlm_path.exists():
        payload = json.loads(vlm_path.read_text(encoding='utf-8'))
        print(json.dumps(payload.get('parsed', {}), ensure_ascii=False, indent=2)[:2500])